# Pipeline de Benchmark do TCC

Roda o sistema de benchmark e documenta o automador completo do TCC:
download/preparação do dataset, split estratificado, treino, inferência,
tabelas LaTeX, figuras PNG, `dataset.md` e `tcc_report.md`.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
while not (ROOT / "app").exists() and ROOT.parent != ROOT:
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print("Projeto:", ROOT)

## 1. Benchmark rápido (sintético) — valida o harness em segundos

In [ ]:
from benchmarks import BenchmarkConfig, run_benchmark

cfg = BenchmarkConfig.quick(
    architectures=["MultiscaleCNN", "SVM"],
    synthetic_n=160,
    snr_levels_db=[20],
    output_dir=str(ROOT / "results" / "notebook_benchmark"),
)
results = run_benchmark(cfg)

for name, r in results["architectures"].items():
    if r.get("status") == "ok":
        c = r["clean"]
        print(f"{name:<16} acc={c['accuracy']*100:5.1f}%  "
              f"AUC={c.get('auc_roc', float('nan')):.3f}  "
              f"conv={r.get('converged')}")
    else:
        print(f"{name:<16} ERRO: {r.get('error')}")

In [ ]:
# Artefatos gerados (tabelas .tex, figuras .png, CSV/JSON).
out = ROOT / "results" / "notebook_benchmark"
for f in sorted(out.rglob("*")):
    if f.is_file():
        print(f.relative_to(out))

## 2. Execução completa do TCC

A célula abaixo é executável, mas fica desligada por padrão para evitar
downloads e treinos longos sem confirmação explícita.

In [ ]:
import subprocess

RUN_FULL_PIPELINE = False
OUTPUT_DIR = ROOT / "results" / "tcc_full_20k"
DATASET_NPZ = ROOT / "app" / "datasets" / "benchmark_audio_raw_20k.npz"

cmd = [
    sys.executable,
    str(ROOT / "scripts" / "run_tcc_pipeline.py"),
    "--tcc-full-dataset",
    "--out", str(OUTPUT_DIR),
    "--npz", str(DATASET_NPZ),
]
print(" ".join(cmd))

if RUN_FULL_PIPELINE:
    subprocess.run(cmd, cwd=ROOT, check=True)
else:
    print("Execução completa desativada. Defina RUN_FULL_PIPELINE = True para rodar.")

Comando equivalente no terminal:

```bash
python scripts/run_tcc_pipeline.py --tcc-full-dataset \
    --out results/tcc_full_20k \
    --npz app/datasets/benchmark_audio_raw_20k.npz
```

Ou o benchmark direto sobre um `.npz` já exportado:

```bash
python scripts/run_benchmark.py --full --dataset app/datasets/brspeech_df.npz
```

Veja `docs/15_BENCHMARK.md` para o mapeamento saída → tabela/figura do TCC.

## 3. Validar e ler artefatos do relatório

In [ ]:
import json
import pandas as pd

try:                                  # display() só existe no kernel IPython
    from IPython.display import display
except Exception:
    display = print

report_dir = ROOT / "results" / "tcc_full_20k"
required = [
    "dataset.md",
    "dataset_manifest.json",
    "results.json",
    "results.csv",
    "tcc_report.md",
    "figures/roc.png",
    "figures/confusion_matrices.png",
    "figures/score_distributions.png",
]
missing = [p for p in required if not (report_dir / p).exists()]
print("Diretório:", report_dir)
print("Ausentes:", missing or "nenhum")

if (report_dir / "results.csv").exists():
    display(pd.read_csv(report_dir / "results.csv"))
if (report_dir / "results.json").exists():
    data = json.loads((report_dir / "results.json").read_text(encoding="utf-8"))
    print("Arquiteturas:", list(data.get("architectures", {}).keys()))